# Qwen3.5-35B-A3B on AWS Trainium 2

This notebook walks through deploying [Qwen3.5-35B-A3B](https://huggingface.co/Qwen/Qwen3.5-35B-A3B) on a **trn2.3xlarge** instance using the NeuronX Distributed Inference (NxDI) framework.

Qwen3.5-35B-A3B is a **native multimodal VLM** with a novel hybrid architecture:
- **30 DeltaNet layers** (linear recurrent attention with gated delta rule)
- **10 standard GQA layers** (with sigmoid output gate, partial RoPE, head_dim=256)
- **All 40 layers use sparse MoE** (256 routed experts, top-8, plus 1 sigmoid-gated shared expert)
- **Built-in vision encoder** (27-layer ViT, 1152 hidden, patch_size=16)

35B total parameters, only 3B active per token.

### What this notebook covers

1. **Setup** -- Install the NxDI fork, download model weights, configure the environment
2. **Text generation** -- Configure, compile, load, and generate text on Neuron
3. **Vision + text generation** -- Compile the vision encoder, process images, run the full VL pipeline
4. **Architecture notes** -- Why this model needs an NxDI fork and what the patches do

### Prerequisites

- **Instance**: `trn2.3xlarge` (4 NeuronCores, 96 GB HBM)
- **AMI**: Deep Learning AMI Neuron (Ubuntu 24.04) -- SDK 2.28
- **Disk**: 300+ GB EBS (model weights are ~67 GB)
- **Time**: ~15 min compile (short context), ~45 min compile (long context), ~5 min for vision encoder

### Estimated wall time

| Step | Time |
|------|------|
| Setup + download model | ~10 min |
| Text model compilation (seq_len=160) | ~12 min |
| Text model load + generation | ~3 min |
| Vision encoder compilation | ~3 min |
| Vision+text generation | ~2 min |
| **Total** | **~30 min** |

---
## 1. Setup

### 1.1 Activate the Neuron venv and set environment variables

Before starting this notebook, activate the pre-installed Neuron virtual environment in your terminal:

```bash
source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/bin/activate
export NEURON_PLATFORM_TARGET_OVERRIDE=trn2
jupyter notebook  # or jupyter lab
```

If you're running in JupyterLab from the DLAMI, the kernel should already be using this venv.

In [ ]:
import os

# Required for NKI v2 kernels on trn2
os.environ["NEURON_PLATFORM_TARGET_OVERRIDE"] = "trn2"

# Remove XLA_DISABLE_FUNCTIONALIZATION if set (interferes with NxDI compilation)
os.environ.pop("XLA_DISABLE_FUNCTIONALIZATION", None)

# Verify Neuron devices are available
!neuron-ls | head -8

### 1.2 Install the NxDI fork

Qwen3.5-35B-A3B requires a fork of NxDI with custom support for:
- DeltaNet linear attention (custom NKI kernels + recurrent state management)
- Sigmoid-gated shared experts (different from NxDI's default additive shared experts)
- Partial RoPE (25% of head_dim), output-gated attention, head_dim=256
- A fix to `hf_adapter.py` for the `tensor_capture_hook` NameError

See Section 4 for full details on what each patch does and why.

In [ ]:
NXDI_REPO = "/home/ubuntu/nxdi-qwen35"

if not os.path.exists(NXDI_REPO):
    !git clone --branch contrib/qwen3.5-35b-a3b \
        https://github.com/jimburtoft/neuronx-distributed-inference.git \
        {NXDI_REPO}
    # Install as editable (--no-deps to avoid overwriting SDK packages)
    !pip install -e {NXDI_REPO} --no-deps
else:
    print(f"NxDI fork already cloned at {NXDI_REPO}")
    !cd {NXDI_REPO} && git pull

### 1.3 Download model weights

The model is ~67 GB (14 safetensors shards). We'll download to a persistent location.

In [ ]:
MODEL_PATH = "/home/ubuntu/models/Qwen3.5-35B-A3B"

if not os.path.exists(os.path.join(MODEL_PATH, "config.json")):
    !huggingface-cli download Qwen/Qwen3.5-35B-A3B --local-dir {MODEL_PATH}
else:
    print(f"Model already downloaded at {MODEL_PATH}")

### 1.4 Install additional dependencies for vision

In [ ]:
!pip install -q Pillow qwen-vl-utils

### 1.5 Add the contrib model source to the Python path

In [ ]:
import sys
CONTRIB_SRC = os.path.join(NXDI_REPO, "contrib/models/Qwen3.5-35B-A3B/src")
if CONTRIB_SRC not in sys.path:
    sys.path.insert(0, CONTRIB_SRC)
print(f"Added {CONTRIB_SRC} to sys.path")

---
## 2. Text Generation

We'll start with text-only generation, which uses the DeltaNet + GQA + MoE text decoder.

### 2.1 Create the inference config

Key configuration choices:
- **TP=4**: One NeuronCore per tensor-parallel rank (trn2.3xlarge has 4 logical cores at LNC=2)
- **BF16**: The model's native precision
- **block_size=2048**: Workaround for an SDK 2.28 blockwise MoE bug -- forces the `forward_all_experts` code path
- **fused_qkv=True**: The GQA layers use fused Q/K/V projections

In [ ]:
import json
import torch
from neuronx_distributed_inference.models.config import MoENeuronConfig
from modeling_qwen35_moe import NeuronQwen35MoeForCausalLM, Qwen35MoeInferenceConfig

# Load the HuggingFace config
with open(os.path.join(MODEL_PATH, "config.json")) as f:
    full_config = json.load(f)
text_config = full_config.get("text_config", full_config)

# NeuronConfig for short context
neuron_config = MoENeuronConfig(
    tp_degree=4,
    max_batch_size=1,
    max_context_length=128,
    max_new_tokens=32,
    on_device_sampling_config=None,
    torch_dtype=torch.bfloat16,
    fused_qkv=True,
    moe_tp_degree=4,
    moe_ep_degree=1,
    blockwise_matmul_config={"block_size": 2048},
)

# Merge HF text config with NeuronConfig
config_dict = dict(text_config)
config_dict["pad_token_id"] = text_config.get("eos_token_id", 248044)
if "rope_parameters" in text_config:
    config_dict["rope_theta"] = text_config["rope_parameters"].get("rope_theta", 10000000)
if config_dict.get("tie_word_embeddings") is None:
    config_dict["tie_word_embeddings"] = False

config = Qwen35MoeInferenceConfig(neuron_config=neuron_config, **config_dict)

print(f"Config created:")
print(f"  seq_len={config.neuron_config.seq_len}")
print(f"  max_context_length={config.neuron_config.max_context_length}")
print(f"  max_new_tokens={config.neuron_config.max_new_tokens}")
print(f"  tp_degree={config.neuron_config.tp_degree}")
print(f"  torch_dtype={config.neuron_config.torch_dtype}")
print(f"  num_hidden_layers={config.num_hidden_layers}")
print(f"  num_experts={config.num_local_experts}")
print(f"  head_dim={config.head_dim}")

### 2.2 Compile the text model

Compilation creates two HLO programs:
- **CTE (Context Encoding / Prefill)**: Processes the full input sequence. DeltaNet layers use the NKI kernel for parallel recurrence.
- **TKG (Token Generation / Decode)**: Processes one token at a time. DeltaNet layers use PyTorch sequential recurrence.

This takes ~12 minutes for seq_len=160. The DeltaNet recurrence is unrolled during HLO generation, so compile time scales with sequence length.

In [ ]:
import time

COMPILED_TEXT_PATH = "/home/ubuntu/compiled_qwen35_text/"

model = NeuronQwen35MoeForCausalLM(model_path=MODEL_PATH, config=config)

if not os.path.exists(os.path.join(COMPILED_TEXT_PATH, "model.pt")):
    os.makedirs(COMPILED_TEXT_PATH, exist_ok=True)
    print("Compiling text model (this takes ~12 minutes)...")
    t0 = time.time()
    model.compile(COMPILED_TEXT_PATH)
    print(f"Compilation complete in {time.time() - t0:.0f}s")
else:
    print(f"Compiled model found at {COMPILED_TEXT_PATH}, skipping compilation")

### 2.3 Load and generate

In [ ]:
from transformers import AutoTokenizer

# Load compiled model onto NeuronCores
print("Loading model onto NeuronCores...")
t0 = time.time()
model.load(COMPILED_TEXT_PATH)
print(f"Loaded in {time.time() - t0:.0f}s")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def greedy_generate(model, input_ids, attention_mask, max_new_tokens=20):
    """Simple greedy generation using model.forward() directly."""
    eos_token_ids = {248044, 248046}  # <|endoftext|>, <|im_end|>
    model.reset()

    batch_size, seq_len = input_ids.shape
    position_ids = torch.arange(seq_len).unsqueeze(0).expand(batch_size, -1)
    all_ids = input_ids.clone()

    # Context encoding (prefill)
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        position_ids=position_ids,
    )
    next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    all_ids = torch.cat([all_ids, next_token], dim=-1)

    # Token generation (autoregressive decode)
    for step in range(max_new_tokens - 1):
        if all(next_token[b, 0].item() in eos_token_ids for b in range(batch_size)):
            break
        cur_pos = seq_len + step + 1
        attention_mask = torch.cat(
            [attention_mask, torch.ones(batch_size, 1, dtype=attention_mask.dtype)], dim=-1
        )
        pos_ids = torch.tensor([[cur_pos - 1]] * batch_size, dtype=torch.long)

        outputs = model(
            input_ids=next_token,
            attention_mask=attention_mask,
            position_ids=pos_ids,
        )
        next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        all_ids = torch.cat([all_ids, next_token], dim=-1)

    return all_ids

In [ ]:
# Test prompts
prompts = [
    "The capital of France is",
    "1 + 1 =",
    "The color of the sky is",
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt", padding=True)
    output_ids = greedy_generate(
        model,
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=20,
    )
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(f"Prompt: {prompt!r}")
    print(f"Output: {text!r}")
    print()

### 2.4 Measure throughput

In [ ]:
# Warmup
inputs = tokenizer("Hello", return_tensors="pt", padding=True)
_ = greedy_generate(model, inputs.input_ids, inputs.attention_mask, max_new_tokens=3)

# Measure
num_tokens = 20
t0 = time.perf_counter()
_ = greedy_generate(model, inputs.input_ids, inputs.attention_mask, max_new_tokens=num_tokens)
elapsed = time.perf_counter() - t0

throughput = num_tokens / elapsed
latency_ms = elapsed / num_tokens * 1000

print(f"Token generation throughput: {throughput:.1f} tok/s")
print(f"Per-token latency: {latency_ms:.1f} ms/tok")

---
## 3. Vision + Text Generation

Qwen3.5-35B-A3B is a **native multimodal model** -- all Qwen3.5 models include a vision encoder (there is no separate `-VL` variant). The architecture includes a 27-layer ViT that projects image patches into the text decoder's embedding space.

The vision pipeline:
1. **Image processing**: HF's `AutoProcessor` resizes and patches the image
2. **Vision encoder**: ViT encodes patches into embeddings (compiled separately with `torch_neuronx.trace()`)
3. **Embedding injection**: Vision embeddings replace image placeholder tokens in the text sequence
4. **mRoPE**: 3D multimodal positional encoding (temporal, height, width) for vision tokens
5. **Text generation**: Standard autoregressive decode with the injected vision context

### 3.1 Recompile text model with vision support

The text model needs to be compiled with a **24-argument trace signature** (vs 7 for text-only) to accept vision embeddings, vision masks, and 3D mRoPE position IDs.

We also need a larger `max_context_length` because image tokens consume sequence positions (a single 224x224 image produces 196 vision tokens after patch merging).

In [ ]:
# Reconfigure for VL: larger context to fit vision tokens
neuron_config_vl = MoENeuronConfig(
    tp_degree=4,
    max_batch_size=1,
    max_context_length=256,  # Larger to fit image tokens + text
    max_new_tokens=32,
    on_device_sampling_config=None,
    torch_dtype=torch.bfloat16,
    fused_qkv=True,
    moe_tp_degree=4,
    moe_ep_degree=1,
    blockwise_matmul_config={"block_size": 2048},
)

config_vl = Qwen35MoeInferenceConfig(neuron_config=neuron_config_vl, **config_dict)

COMPILED_VL_TEXT_PATH = "/home/ubuntu/compiled_qwen35_vl_text/"

# The VL model needs a fresh model instance
model_vl_text = NeuronQwen35MoeForCausalLM(model_path=MODEL_PATH, config=config_vl)

if not os.path.exists(os.path.join(COMPILED_VL_TEXT_PATH, "model.pt")):
    os.makedirs(COMPILED_VL_TEXT_PATH, exist_ok=True)
    print("Compiling text model with vision support (~12 min)...")
    t0 = time.time()
    model_vl_text.compile(COMPILED_VL_TEXT_PATH)
    print(f"Compilation complete in {time.time() - t0:.0f}s")
else:
    print(f"Compiled VL text model found at {COMPILED_VL_TEXT_PATH}")

### 3.2 Compile the vision encoder

The vision encoder is compiled separately using `torch_neuronx.trace()` (not the NxDI `compile()` path). It's a standalone ViT that runs on a single NeuronCore.

In [ ]:
COMPILED_VISION_DIR = "/home/ubuntu/compiled_qwen35_vision/"
COMPILED_VISION_PATH = os.path.join(COMPILED_VISION_DIR, "vision_encoder.pt")

if not os.path.exists(COMPILED_VISION_PATH):
    os.makedirs(COMPILED_VISION_DIR, exist_ok=True)
    print("Compiling vision encoder (~3 min)...")
    # The script reads paths from environment variables
    os.environ["QWEN35_MODEL_PATH"] = MODEL_PATH
    os.environ["QWEN35_VISION_COMPILED_PATH"] = COMPILED_VISION_DIR
    !cd {CONTRIB_SRC} && python compile_vision_encoder.py
else:
    print(f"Compiled vision encoder found at {COMPILED_VISION_PATH}")

### 3.3 Load the full VL model

In [ ]:
from modeling_qwen35_moe_vl import NeuronQwen35MoeVLForCausalLM, Qwen35MoeVLInferenceConfig

# Create VL config
vl_config = Qwen35MoeVLInferenceConfig(
    text_config=None,
    vision_config=full_config.get("vision_config", {}),
    image_token_id=full_config.get("image_token_id", 248056),
    video_token_id=full_config.get("video_token_id", 248057),
    vision_start_token_id=full_config.get("vision_start_token_id", 248053),
    vision_end_token_id=full_config.get("vision_end_token_id", 248054),
    spatial_merge_size=full_config.get("vision_config", {}).get("spatial_merge_size", 2),
    vision_seq_len_buckets=[256, 1024, 4096],
)

# Create VL model (wraps text decoder + vision encoder)
vl_model = NeuronQwen35MoeVLForCausalLM(
    model_path=MODEL_PATH,
    text_config=config_vl,
    vision_config=vl_config,
)

# Load text model
print("Loading text model...")
t0 = time.time()
vl_model.text_model.load(COMPILED_VL_TEXT_PATH)
print(f"Text model loaded in {time.time() - t0:.0f}s")

# Load vision model
print("Loading vision model...")
t0 = time.time()
vl_model.vision_model_wrapper.load_compiled(COMPILED_VISION_PATH)
vl_model.vision_model_wrapper.load_vision_weights_from_hf(MODEL_PATH)
print(f"Vision model loaded in {time.time() - t0:.0f}s")

### 3.4 Text-only sanity check

First verify that the 24-arg VL model still handles text-only prompts correctly.

In [ ]:
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")

generated = vl_model.generate(
    input_ids=inputs["input_ids"],
    max_new_tokens=16,
)

output = tokenizer.decode(generated[0], skip_special_tokens=True)
print(f"Prompt: {prompt!r}")
print(f"Output: {output!r}")

### 3.5 Vision + text generation

Now let's process a real image. We'll use a simple test image first, then you can try your own.

In [ ]:
from PIL import Image
from transformers import AutoProcessor
import numpy as np

processor = AutoProcessor.from_pretrained(MODEL_PATH)

# Create a test image: red square
img_array = np.zeros((224, 224, 3), dtype=np.uint8)
img_array[:, :, 0] = 255  # Red channel
test_image = Image.fromarray(img_array)
test_image

In [ ]:
# Prepare multimodal input using HF processor
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "What color is this image?"},
        ],
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[test_image], return_tensors="pt")

print(f"input_ids shape: {inputs['input_ids'].shape}")
print(f"pixel_values shape: {inputs.get('pixel_values').shape}")
print(f"image_grid_thw: {inputs.get('image_grid_thw')}")

# Count image tokens in the sequence
n_image_tokens = (inputs["input_ids"] == 248056).sum().item()
print(f"Image placeholder tokens: {n_image_tokens}")

In [ ]:
# Run vision + text generation
t0 = time.time()
generated = vl_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs.get("attention_mask", torch.ones_like(inputs["input_ids"])),
    pixel_values=inputs.get("pixel_values"),
    image_grid_thw=inputs.get("image_grid_thw"),
    max_new_tokens=32,
)
gen_time = time.time() - t0

output = processor.decode(generated[0], skip_special_tokens=True)
print(f"Output: {output!r}")
print(f"Generation time: {gen_time:.2f}s")

### 3.6 Try with a real image (optional)

Download an image from the web and ask the model about it.

In [ ]:
import requests
from io import BytesIO

# Download a sample image (Eiffel Tower)
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a8/Tour_Eiffel_Wikimedia_Commons.jpg/800px-Tour_Eiffel_Wikimedia_Commons.jpg"
try:
    response = requests.get(url, timeout=10)
    real_image = Image.open(BytesIO(response.content)).convert("RGB")
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": "What is shown in this image?"},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[real_image], return_tensors="pt")
    
    generated = vl_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask", torch.ones_like(inputs["input_ids"])),
        pixel_values=inputs.get("pixel_values"),
        image_grid_thw=inputs.get("image_grid_thw"),
        max_new_tokens=32,
    )
    output = processor.decode(generated[0], skip_special_tokens=True)
    print(f"Output: {output!r}")
    display(real_image.resize((400, 300)))
except Exception as e:
    print(f"Could not download image: {e}")
    print("Try with your own image by replacing the URL above.")

---
## 4. Architecture Notes

### Why does this model need an NxDI fork?

Qwen3.5-35B-A3B introduces several architectural novelties that are not yet supported in the upstream NxDI framework. Here's what each patch does:

| Component | What's Different | NxDI Patch |
|-----------|-----------------|------------|
| **DeltaNet linear attention** | Gated delta rule recurrence instead of softmax attention. Requires custom NKI v2 kernel, recurrent state buffers, and `input_output_aliases` for state carry-over. | Entirely custom: `NeuronGatedDeltaNet` + `nki_deltanet.py` |
| **Sigmoid-gated shared expert** | Qwen3.5 gates the shared expert output with `sigmoid(gate(x)) * expert(x)`. NxDI's default adds shared expert output directly. | Custom `SigmoidGatedSharedExperts` wrapper |
| **Output gate in GQA** | Attention output is gated: `sigmoid(gate_proj(x)) * attn_output`. The gate weights are interleaved with query weights in the HF checkpoint. | Custom `NeuronQwen35Attention` + state dict deinterleaving |
| **Partial RoPE (25%)** | Only the first 64 of 256 head dimensions get rotary embedding. | Override `apply_rotary_embedding()` |
| **head_dim=256** | NxDI's NKI flash attention kernel asserts head_dim<=128. | `perform_prefill()` override disables NKI kernel, uses PyTorch softmax path |
| **RMSNorm +1.0** | Qwen3.5 uses `(1+weight) * norm(x)` instead of `weight * norm(x)`. | State dict converter adds 1.0 to all RMSNorm weights |
| **MoE blockwise bug** | SDK 2.28's `forward_blockwise` path produces incorrect output on trn2. | Config workaround: set `block_size` large enough to force `forward_all_experts` |
| **hf_adapter.py NameError** | `tensor_capture_hook` referenced but never extracted from kwargs. | 2-line fix: `kwargs.pop()` + remove from `model_inputs` |
| **Vision (mRoPE + scatter)** | 3D multimodal position encoding + vision embedding injection into text sequence. 24-argument trace signature. | Custom `_get_model_outputs()`, `input_generator()`, `pad_inputs()` overrides |

### Key SDK 2.28 workarounds

- **`NEURON_PLATFORM_TARGET_OVERRIDE=trn2`**: Required for NKI v2 `@nki.jit` kernels
- **`XLA_DISABLE_FUNCTIONALIZATION` removal**: Set by `torch_neuronx` import, interferes with NxDI compilation
- **`--enable-saturate-infinity`**: Compiler flag to handle infinity in attention masks safely
- **`-65504.0` instead of `-inf`**: For vision attention masks, avoids BF16 NaN propagation
- **`--internal-enable-dge-levels vector_dynamic_offsets`**: Required for DeltaNet's dynamic index operations

### Performance characteristics

| Metric | Value (trn2.3xlarge, BS=1) |
|--------|---------------------------|
| TKG throughput | ~54 tok/s |
| TKG latency | ~18 ms/tok |
| TTFT (seq_len=128) | ~1,050 ms |
| TTFT (seq_len=2048) | ~10,100 ms |
| Compile time (seq_len=160) | ~12 min |
| Compile time (seq_len=2176) | ~42 min |
| Model load time | ~95-195 s |

TTFT at longer sequences is dominated by the PyTorch softmax attention fallback for head_dim=256 layers. A custom NKI flash attention kernel for head_dim=256 (`nki_flash_attn_d256.py`) is included in the contrib source, but is currently ~2.4x slower due to layout conversion overhead.

---
## Cleanup

The compiled models are saved to disk and can be reused across sessions. To free NeuronCore memory without deleting compiled artifacts:

In [ ]:
# Uncomment to free NeuronCore memory
# del model
# del vl_model
# import gc; gc.collect()
# import torch; torch.cuda.empty_cache()  # Also clears Neuron device memory

print("Done! Compiled models saved at:")
print(f"  Text-only: {COMPILED_TEXT_PATH}")
print(f"  VL text:   {COMPILED_VL_TEXT_PATH}")
print(f"  Vision:    {COMPILED_VISION_PATH}")